# A seed is worth more than two layers: the N=48 leak-free ladder has no usable rung-to-rung resolution

**Jobs `50691432` (L=8, seed 1, COMPLETED 0:0, 7 h 30 m) and `50691430` (L=10, seed 1), both on
`sganguli`, commit at submit `a10a098`. Plus `50728837` (depth re-eval at `n_pool` 6000, no
retraining, exit 1:0 after the science completed -- see §7).**

Finding 40 put **L=10 (0.8007) below L=8 (0.8660)** on the shared n=1500 pool, paired
**McNemar exact p = 5.4e-09**. That p compares two *checkpoints* on the same instances. It says
nothing about two *training procedures*, and **no rung of this ladder had ever been run twice**.
H1 said the inversion was optimization noise; H2 said depth genuinely hurts here. These two jobs
retrain L=8 and L=10 with `--seed 1` and change nothing else.

**The answer is H1, and far more of it than H1 needed.** At fixed architecture the two seeds differ
by a spread that is *larger than the depth gaps the ladder is built out of*. The pre-registration
named 0.065 as the threshold above which "every single-seed L\* statement in STATUS (findings
11/14/22/33/38/40) is uninterpretable without replicates", and added: *"That is the most
consequential outcome of the two and it is not the one I expect."* It is what happened.

The second job is the useful counterpoint. It re-evaluated the six **seed-0** checkpoints on a 4x
larger depth pool and showed the **sampling** error bar on L\* is now small -- which is exactly what
isolates the seed as the term that is actually uncontrolled.

**Finding 40 is withdrawn.** Re-run within seed 1, the comparison it rests on goes the *other way*:
L=10 - L=8 = **+0.1920** at seed 1 against **-0.0653** at seed 0. The inversion was a draw, not a
property of depth 10.

**Nothing improved tonight.** The best N=48 leak-free cell is still L=12 seed 0 at **0.9780**
[0.9693, 0.9843], unchanged since finding 38. The two new cells land on opposite sides of their
seed-0 twins (L=8 s1 **-0.1627**, L=10 s1 **+0.0947**), which is itself the point: the seed is not
a bias to subtract, it is noise in both directions. The deliverable is an error bar, not a number.

In [ ]:
import glob, io, json, math, os, re
from statistics import NormalDist
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt

S = os.environ["SCRATCH"] + "/spin_gnn"
DIRS = [f"{S}/sudoku_n48_leakfree_lstar_2026-10-29",   # 50099945: L=6, 4, 3
        f"{S}/sudoku_n48_leakfree_lstar_2026-10-31",   # 50527633 L=12, 50527634 L=8
        f"{S}/sudoku_n48_leakfree_L10_2026-10-31",     # 50607549: L=10
        f"{S}/sudoku_n48_leakfree_seed1_2026-11-01"]   # 50691430/50691432: seed 1
BIGPOOL = f"{S}/sudoku_n48_leakfree_bigpool_2026-11-01"
DEPTH_GLOBS = ["depth_lstar_*.csv", "depth_L10_*.csv", "depth_seed1_*.csv"]
GATE, LADDER = 0.90, [12, 10, 8, 6, 4, 3]
NORMAL_Q = NormalDist().inv_cdf        # stdlib: keeps this notebook off py-scipy

def cell_of(ckpt):
    '''(num_layers, seed) from a checkpoint tag. Seed 0 tags carry no _sNN.'''
    name = str(ckpt)
    L = int(re.search(r"_L(\d+)_", name).group(1))
    m = re.search(r"_s(\d+)(?:\b|_|$)", name)
    return L, (int(m.group(1)) if m else 0)

def lab(c):
    L, s = cell_of(c); return f"L={L} s{s}"

def wilson(k, n, z=1.96):
    p, d = k / n, 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - h) / d, (c + h) / d)

def mcnemar_exact(b, c):
    '''Two-sided exact McNemar on discordant counts b and c.'''
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

pool_csvs, bin_csvs, npzs = [], [], []
for d in DIRS:
    pool_csvs += sorted(glob.glob(f"{d}/sharedpool_n1500_*.csv"))
    npzs += sorted(glob.glob(f"{d}/sharedpool_n1500_*_per_instance.npz"))
    for pat in DEPTH_GLOBS:
        bin_csvs += sorted(glob.glob(f"{d}/{pat}"))
big_csvs = sorted(glob.glob(f"{BIGPOOL}/depth_bigpool_*.csv"))
print(f"{len(pool_csvs)} pool CSVs, {len(bin_csvs)} depth CSVs, {len(npzs)} npz, "
      f"{len(big_csvs)} big-pool CSVs")

## 1. Provenance, config and seed

The claim "`--seed` is the only difference" is the load-bearing one for the whole result, so it is
checked against each checkpoint's own `config.json` rather than against the sbatch scripts.

One honest asterisk, finding 23's: `build_instances(cfg, n_train, seed_start=seed)` shifts the
training set by **one index**, so 239,999 of the 240,000 training instances are shared between the
two seeds. That is deliberate -- every rung of the existing ladder trained on seed 0's instances, so
the only thing this experiment varies is init and batch order, which is precisely the quantity the
single-seed ladder was implicitly assuming was negligible. `model_max_len` is derived from the
realized data, so the one swapped instance can move it by a token or two and with it a handful of
positional-embedding parameters.

In [ ]:
CELLDIRS = {
    (12, 0): f"{S}/sudoku_n48_leakfree_lstar_2026-10-31/L12",
    (10, 0): f"{S}/sudoku_n48_leakfree_L10_2026-10-31/L10",
    ( 8, 0): f"{S}/sudoku_n48_leakfree_lstar_2026-10-31/L08",
    ( 6, 0): f"{S}/sudoku_n48_leakfree_lstar_2026-10-29/L06",
    (10, 1): f"{S}/sudoku_n48_leakfree_seed1_2026-11-01/L10",
    ( 8, 1): f"{S}/sudoku_n48_leakfree_seed1_2026-11-01/L08",
}
KEYS = ("d_model", "nhead", "dim_feedforward", "n_train_instances", "epochs", "lr",
        "lr_schedule", "warmup_epochs", "batch_size", "shuffle_adj", "shuffle_events",
        "rho", "rho_max", "n_nodes", "degree", "n_colors", "norm_first")

rows, base, hist = [], None, {}
for (L, seed), d0 in sorted(CELLDIRS.items(), reverse=True):
    tag = f"d0256_h04_L{L:02d}_ff01024_lr1e-03_pre_shuf" + (f"_s{seed:02d}" if seed else "")
    d = f"{d0}/{tag}"
    if not os.path.isdir(d):
        print(f"  (L={L} s{seed}: checkpoint dir not present yet, skipped)")
        continue
    cfg = json.load(open(f"{d}/config.json"))
    h = json.load(open(f"{d}/history.json"))
    v = [e["nll"] for e in h]; a = [e["accuracy"] for e in h]
    hist[(L, seed)] = dict(curve=v, val_nll=v[-1], acc=a[-1])
    assert cfg["seed"] == seed, f"{tag}: config seed {cfg['seed']} != {seed}"
    assert cfg["num_layers"] == L
    rows.append(dict(cell=f"L={L} s{seed}", L=L, seed=seed, epochs=len(v),
                     final_val_nll=v[-1], min_val_nll=min(v), final_tok_acc=a[-1],
                     max_len=cfg.get("model_max_len")))
    cur = {k: cfg[k] for k in KEYS}
    if base is None:
        base = cur
    else:
        delta = {k: (base[k], cur[k]) for k in KEYS if base[k] != cur[k]}
        assert not delta, f"{tag} differs beyond num_layers/seed: {delta}"
print("\nconfig check: num_layers and seed are the ONLY differences across all cells")
print(f"n_train {base['n_train_instances']}, epochs {base['epochs']}, ff {base['dim_feedforward']}, "
      f"d_model {base['d_model']}, lr {base['lr']}, shuffle_adj {base['shuffle_adj']}")
prov = pd.DataFrame(rows).sort_values(["L", "seed"], ascending=[False, True])
print()
print(prov.round(4).to_string(index=False))

## 2. The metric of record: the shared pool, n=1500

`sudoku_agg_eval`, `seed_start=5e10`, disjoint from every training set, instance *i* drawn from
`seed_start + i` alone. `seed` is **not** in `INSTANCE_KEYS`, so a seed-1 checkpoint is scored on
exactly the same 1500 instances as every seed-0 cell and every contrast below is paired instance by
instance.

Each job re-evaluated the earlier cells, so several checkpoints were scored two or three times.
Those repeats are asserted to agree **instance for instance** before anything is pooled -- that
assertion is pre-registration (d), and it is the only determinism check this eval path gets.

In [ ]:
solved, seen_in, seed_starts, n_repeat = {}, {}, set(), 0
for p in npzs:
    job = re.search(r"_(\d+)_per_instance", p).group(1)
    z = np.load(p, allow_pickle=True)
    seed_starts.add(int(z["seed_start"]))
    for i, name in enumerate(z["names"]):
        key, v = lab(str(name)), np.asarray(z["solved"][i], bool)
        if key in solved:
            n_repeat += 1
            agree = bool((solved[key] == v).all())
            print(f"  repeat {n_repeat:2d}: {key:<8} job {job} vs {seen_in[key]} -> "
                  f"{'IDENTICAL on all %d' % v.size if agree else 'DISAGREE'}")
            assert agree, f"{key} disagrees between {job} and {seen_in[key]}"
        else:
            solved[key], seen_in[key] = v, job
assert len(seed_starts) == 1, f"pools differ: {seed_starts}"
print(f"\nseed_start {seed_starts.pop()}; {n_repeat} repeat evaluations, all identical")

tab = []
for L in LADDER:
    for seed in (0, 1):
        k = f"L={L} s{seed}"
        if k not in solved:
            continue
        v = solved[k]
        lo, hi = wilson(int(v.sum()), v.size)
        tab.append(dict(cell=k, L=L, seed=seed, n=v.size, solve_rate=v.mean(),
                        wilson_lo=lo, wilson_hi=hi,
                        val_nll=hist.get((L, seed), {}).get("val_nll", float("nan")),
                        tok_acc=hist.get((L, seed), {}).get("acc", float("nan"))))
tab = pd.DataFrame(tab)
print()
print(tab.round(4).to_string(index=False))

## 3. The headline: the seed spread, paired

`s_L = |pool(L, seed 1) - pool(L, seed 0)|`, computed on the same 1500 instances, so this is a
within-instance paired contrast and not a difference of two independent rates. The two depth
contrasts the ladder is built out of are printed beside it on exactly the same footing.

In [ ]:
def paired(a, b):
    x, y = solved[a], solved[b]
    w, l = int((x & ~y).sum()), int((~x & y).sum())
    return float(x.mean() - y.mean()), w, l, mcnemar_exact(w, l)

CON = [("seed spread at L=8",    "L=8 s1",  "L=8 s0",  "noise"),
       ("seed spread at L=10",   "L=10 s1", "L=10 s0", "noise"),
       ("depth gap, finding 38", "L=12 s0", "L=8 s0",  "signal"),
       ("finding-40 inversion",  "L=10 s0", "L=8 s0",  "signal"),
       ("depth, within seed 1",  "L=10 s1", "L=8 s1",  "signal")]
rows = []
for name, a, b, kind in CON:
    if a not in solved or b not in solved:
        print(f"  {name}: {a} or {b} missing, skipped")
        continue
    d, w, l, p = paired(a, b)
    rows.append(dict(contrast=name, cells=f"{a} - {b}", kind=kind, delta=d,
                     abs_delta=abs(d), won=w, lost=l, mcnemar_p=p))
con = pd.DataFrame(rows)
print(con.round(4).to_string(index=False))

noise = con[con.kind == "noise"].abs_delta.max()
sig = con[(con.kind == "signal") & con.contrast.isin(
    ["depth gap, finding 38", "finding-40 inversion"])].abs_delta
print(f"\nmax seed spread           {noise:.4f}")
print(f"L=12 - L=8 depth gap      {sig.max():.4f}   ratio spread/gap  {noise/sig.max():.2f}")
print(f"finding-40 inversion      {sig.min():.4f}   ratio spread/gap  {noise/sig.min():.2f}")
print("\nThe seed spread exceeds BOTH depth contrasts." if noise > sig.max() else
      "\nThe seed spread sits between the two depth contrasts.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for seed, col, ls in ((0, "#2b6cb0", "-"), (1, "#c53030", "--")):
    sub = tab[tab.seed == seed].sort_values("L")
    if sub.empty:
        continue
    ax1.errorbar(sub.L, sub.solve_rate,
                 yerr=[sub.solve_rate - sub.wilson_lo, sub.wilson_hi - sub.solve_rate],
                 marker="o", ms=8, lw=2, capsize=4, color=col, ls=ls, label=f"seed {seed}")
ax1.axhline(GATE, color="0.6", ls=":")
ax1.set_xlabel("num_layers"); ax1.set_ylabel("solve_rate, shared pool n=1500")
ax1.set_title("The same ladder, two draws"); ax1.set_xticks(LADDER)
ax1.legend(loc="lower right"); ax1.grid(alpha=0.3)

bars = con[con.contrast != "depth, within seed 1"]
cols = ["#c53030" if k == "noise" else "#2b6cb0" for k in bars.kind]
ax2.bar(range(len(bars)), bars.abs_delta, color=cols)
ax2.set_xticks(range(len(bars)))
ax2.set_xticklabels([c.replace(" at ", "\n").replace(", ", "\n") for c in bars.contrast], size=9)
ax2.set_ylabel("|delta solve_rate|"); ax2.set_title("noise (red) vs signal (blue)")
ax2.grid(alpha=0.3, axis="y")
for i, v in enumerate(bars.abs_delta):
    ax2.text(i, v + 0.003, f"{v:.4f}", ha="center", size=9)
plt.tight_layout(); plt.show()

## 4. Where the seed costs its points: the depth bins

`sudoku_depth_eval` at rho=0.60, realized density pinned in every bin, so "deeper" is not "sparser".
If the seed shifted the curve uniformly it would be a nuisance; it does not. It costs almost nothing
on the shallow bins and most of its points on the deep ones -- which is exactly the axis L\* is
defined on, and why a seed can move L\* by more than a layer can.

In [ ]:
dep = pd.concat([pd.read_csv(p) for p in bin_csvs], ignore_index=True)
dep["L"] = [cell_of(c)[0] for c in dep.checkpoint]
dep["seed"] = [cell_of(c)[1] for c in dep.checkpoint]
dep = dep.drop_duplicates(subset=["L", "seed", "rho_requested", "T"])

for rho in sorted(dep.rho_requested.unique(), reverse=True):
    sub = dep[dep.rho_requested == rho]
    piv = sub.pivot_table(index=["L", "seed"], columns="T", values="solve_rate")
    cnt = sub.pivot_table(index=["L", "seed"], columns="T", values="n").iloc[0]
    print(f"\n### rho={rho:.2f}   bin sizes n = {dict(cnt.astype(int))}")
    piv["bins_cleared"] = (piv >= GATE).sum(axis=1).astype(str) + "/" + str(piv.shape[1])
    print(piv.sort_index(ascending=[False, True]).round(4).to_string())
    d = sub.groupby(["L", "seed"]).realized_rho.agg(["min", "max"])
    print(f"realized_rho across all cells/bins: {d['min'].min():.6f} - {d['max'].max():.6f}")

# What the gate count would say if seed 1 had been the draw on record. This is
# the L* claim restated per seed, which is the whole reason the spread matters.
print("\n### the gate verdict at rho=0.60, by seed (bins cleared out of 6)")
sub = dep[dep.rho_requested == 0.60]
for L in (10, 8):
    for s in (0, 1):
        r = sub[(sub.L == L) & (sub.seed == s)]
        if r.empty:
            continue
        worst = r.loc[r.solve_rate.idxmin()]
        print(f"  L={L:>2} s{s}: {(r.solve_rate >= GATE).sum()}/6 cleared, "
              f"worst bin T={int(worst['T'])} at {worst.solve_rate:.4f} "
              f"({worst.solve_rate - GATE:+.4f} vs gate, n={int(worst['n'])})")
print("  L=10 s1 misses the 0.90 gate in ONE bin and by a margin smaller than one instance in it;")
print("  L=10 s0 misses in four. Same architecture, same data, different --seed.")

In [ ]:
cells = [L for L in (8, 10) if ((dep.L == L) & (dep.seed == 1)).any()]
fig, axes = plt.subplots(1, max(len(cells), 1), figsize=(6.5 * max(len(cells), 1), 5),
                         squeeze=False)
for ax, L in zip(axes[0], cells):
    for seed, col, ls in ((0, "#2b6cb0", "-"), (1, "#c53030", "--")):
        r = dep[(dep.L == L) & (dep.seed == seed) & (dep.rho_requested == 0.60)].sort_values("T")
        if r.empty:
            continue
        ax.plot(r["T"], r.solve_rate, marker="o", ms=7, lw=2, color=col, ls=ls,
                label=f"seed {seed}")
    ax.axhline(GATE, color="0.6", ls=":")
    ax.set_title(f"L={L}, rho=0.60"); ax.set_xlabel("T (DD rounds required)")
    ax.set_ylabel("solve_rate in bin"); ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Pre-registration scoring

Quoted verbatim from `scripts/slurm/sudoku2_n48_seed1_replicate.sbatch`, written before either cell
was submitted.

> **(a) HEADLINE, the H1/H2 discriminator, moderate confidence.** Write
> `s_L = |pool(L, seed 1) - pool(L, seed 0)|`. Prediction: `max(s_8, s_10) >= 0.040`.
> FALSIFIER: `s_8 < 0.040 AND s_10 < 0.040`. *"If instead either s exceeds 0.065, the spread is as
> large as the depth effects this ladder measures and EVERY single-seed L\* statement in STATUS
> (findings 11/14/22/33/38/40) is uninterpretable without replicates. That is the most consequential
> outcome of the two and it is not the one I expect."*
>
> **(b) ORDERING, near a coin flip even under H1.** `D_1 = pool(L=10, s1) - pool(L=8, s1)`.
> Prediction `D_1 > 0`. FALSIFIER: `D_1 < 0` with McNemar p < 0.05.
> *Scoring note, fixed in advance:* `D_1 < 0` together with a large `s` does **not** establish H2;
> it establishes that single-seed rungs are not separable here.
>
> **(c) MECHANISM, high confidence.** (i) within seed 1, `sign(nll_10 - nll_8)` is opposite to
> `sign(D_1)`; (ii) `|D pool| / |D val_nll|` stays in [10, 60] points per 0.001 nll.
>
> **(d) CONTROLS, low confidence that anything moves.** Realized density reproduces 0.5836 to within
> 0.001; repeats agree instance for instance; `truncated_rate <= 0.002` and
> `outcome_accuracy >= 0.99` for both new cells.

In [ ]:
score = {}
s8 = abs(paired("L=8 s1", "L=8 s0")[0]) if "L=8 s1" in solved else float("nan")
s10 = abs(paired("L=10 s1", "L=10 s0")[0]) if "L=10 s1" in solved else float("nan")
smax = np.nanmax([s8, s10])
score["(a) max spread >= 0.040"] = (
    "CONFIRMED" if smax >= 0.040 else "FALSIFIED",
    f"s_8={s8:.4f}, s_10={s10:.4f}, max={smax:.4f}"
    + ("  -- ABOVE the 0.065 alarm threshold" if smax >= 0.065 else ""))

if "L=10 s1" in solved and "L=8 s1" in solved:
    D1, w1, l1, p1 = paired("L=10 s1", "L=8 s1")
    score["(b) D_1 > 0"] = ("CONFIRMED" if D1 > 0 else "FALSIFIED",
                            f"D_1={D1:+.4f}, discordant {w1}-{l1}, p={p1:.2e}")
    dn = hist[(10, 1)]["val_nll"] - hist[(8, 1)]["val_nll"]
    ok_i = (dn < 0) == (D1 > 0)
    sens = abs(D1) / abs(dn) if dn else float("nan")
    score["(c) val_nll concordance + sensitivity in [10, 60]"] = (
        "CONFIRMED" if (ok_i and 10 <= sens <= 60) else "FALSIFIED",
        f"d val_nll={dn:+.4f}, d pool={D1:+.4f}, signs "
        f"{'agree' if ok_i else 'DISAGREE'}; |d pool|/|d val_nll| = {sens:.1f} "
        f"(seed-0 pairs give 28 and 25)")
    # the same model applied to the seed spread itself, as an out-of-sample check
    for L in (8, 10):
        dnl = hist[(L, 1)]["val_nll"] - hist[(L, 0)]["val_nll"]
        obs = paired(f"L={L} s1", f"L={L} s0")[0]
        print(f"  seed spread at L={L}: d val_nll {dnl:+.4f} -> predicted d pool "
              f"{-25.2*dnl:+.4f} at the seed-0 sensitivity, observed {obs:+.4f}")

pool = pd.concat([pd.read_csv(p) for p in pool_csvs], ignore_index=True)
pool["cell"] = [lab(c) for c in pool.checkpoint]
new = pool[pool.cell.isin(["L=8 s1", "L=10 s1"])].drop_duplicates("cell")
rho_min = dep[dep.rho_requested == 0.60].realized_rho.min()
rho_max = dep[dep.rho_requested == 0.60].realized_rho.max()
ok_d = (abs(rho_min - 0.5836) < 0.001 and abs(rho_max - 0.5836) < 0.001
        and (new.truncated_rate <= 0.002).all() and (new.outcome_accuracy >= 0.99).all())
score["(d) controls"] = ("CONFIRMED" if ok_d else "FALSIFIED",
                         f"realized_rho {rho_min:.6f}-{rho_max:.6f}; "
                         f"truncated_rate <= {new.truncated_rate.max():.4f}; "
                         f"outcome_accuracy >= {new.outcome_accuracy.min():.4f}; "
                         f"{n_repeat} repeats identical")

for k, (v, detail) in score.items():
    print(f"{v:>10}  {k}\n            {detail}")

## 6. What (a) costs: every L\* on this ladder

The spread is not a nuisance parameter -- it is bigger than the quantity being measured. Below, each
depth contrast the ladder rests on is divided by the measured seed spread. A ratio below 1 means the
contrast is smaller than the noise on a single rung, i.e. that pair of rungs is **not separable at
one seed**.

In [ ]:
rows = []
for a in LADDER:
    for b in LADDER:
        if a <= b or f"L={a} s0" not in solved or f"L={b} s0" not in solved:
            continue
        d, w, l, p = paired(f"L={a} s0", f"L={b} s0")
        rows.append(dict(pair=f"L={a} vs L={b}", delta=d, mcnemar_p=f"{p:.1e}",
                         ratio_to_seed_spread=abs(d) / smax,
                         separable_at_one_seed="yes" if abs(d) / smax > 1 else "NO"))
sep = pd.DataFrame(rows).sort_values("ratio_to_seed_spread", ascending=False)
print(sep.round(4).to_string(index=False))
print(f"\n{(sep.separable_at_one_seed == 'NO').sum()} of {len(sep)} depth pairs on this ladder are "
      f"smaller than the seed spread ({smax:.4f}).")

## 7. The other error bar: sampling, measured by `50728837`

The same six **seed-0** checkpoints, re-evaluated at `--n-pool 6000 --min-per-bin 150`. No
retraining. Because `build_instances` draws instance *i* from `seed_start + i` alone, the 6000-pool
**nests** the 1500-pool exactly, so this supersedes the smaller evaluation rather than competing
with it.

**The job exited 1:0 and the science is unaffected.** Its `module load` line omitted `py-pandas`;
the 64-minute eval ran to completion and wrote all 60 rows, then the final bin-comparison block died
with `ModuleNotFoundError: No module named 'pandas'`. Fixed in the sbatch (commit `0bc1f32`) and the
comparison was run on a login node rather than spending another GPU-hour. One fix attempt, no
resubmission.

Pre-registered for that job:
> **(A) HEADLINE.** On the same six rho=0.60 bins, L=12's worst-bin Bonferroni 95% lower bound
> reaches >= 0.90. FALSIFIER: the bound stays below 0.90 with n >= 250.
> **(B) BIN SET, genuinely uncertain.** At rho=0.60 at least one new bin (T >= 9) reaches 150.
> *Scoring rule fixed in advance: (A) is scored ONLY on the old six bins.*
> **(C) CONTROL, high confidence.** Every shared bin has `n_6000/n_1500` in [3.0, 5.0] and every
> `solve_rate` within 3 binomial SE of its n=1500 value.
> **(D)** L\*(rho=0.75) = 8 still clears and its lower bound rises.

In [ ]:
big = pd.concat([pd.read_csv(p) for p in big_csvs], ignore_index=True)
big["L"] = [cell_of(c)[0] for c in big.checkpoint]
big["seed"] = [cell_of(c)[1] for c in big.checkpoint]
big = big.drop_duplicates(subset=["L", "seed", "rho_requested", "T"])

# (C) nesting control, on every shared (cell, rho, T)
m = dep.merge(big, on=["L", "seed", "rho_requested", "T"], suffixes=("_1500", "_6000"))
m["ratio"] = m.n_6000 / m.n_1500
se = np.sqrt(m.solve_rate_1500 * (1 - m.solve_rate_1500) / m.n_1500).replace(0, np.nan)
m["z"] = (m.solve_rate_6000 - m.solve_rate_1500) / se
print(f"(C) shared bins {len(m)}; size ratio {m.ratio.min():.2f}-{m.ratio.max():.2f} "
      f"(want [3.0, 5.0]); max |z| on solve_rate {m.z.abs().max():.2f} (want <= 3)")
print(f"    -> {'CONFIRMED' if m.ratio.between(3, 5).all() and m.z.abs().max() <= 3 else 'FALSIFIED'}")

print()
for rho in sorted(big.rho_requested.unique(), reverse=True):
    o = sorted(dep[(dep.rho_requested == rho) & (dep.seed == 0)]["T"].unique())
    n = sorted(big[big.rho_requested == rho]["T"].unique())
    print(f"(B) rho={rho:.2f}: bins {o} -> {n}  {'(new bin admitted)' if n != o else '(unchanged)'}")

In [ ]:
def gate_audit(df, label):
    '''Bonferroni-corrected Wilson lower bound on the bin that binds AT 95%.

    L* is an INTERSECTION hypothesis over bins and an ABSOLUTE threshold on ONE
    cell, so -- unlike every paired contrast above -- it gets no power from the
    shared instances. Its precision is set by bin size alone, and because n
    varies several-fold across bins the bin that binds at 95% is often not the
    one that binds on the point estimate.'''
    out = []
    for rho in sorted(df.rho_requested.unique(), reverse=True):
        sub = df[df.rho_requested == rho]
        nb = sub["T"].nunique()
        z = NORMAL_Q(1 - 0.05 / (2 * nb))
        for (L, seed), r in sub.groupby(["L", "seed"]):
            if r["T"].nunique() < nb:
                continue
            los = [wilson(int(round(rr.solve_rate * rr["n"])), int(rr["n"]), z)[0]
                   for _, rr in r.iterrows()]
            worst = r.iloc[int(np.argmin(los))]
            clears = int((r.solve_rate >= GATE).sum())
            out.append(dict(source=label, rho=rho, cell=f"L={L} s{seed}",
                            bins=f"{clears}/{nb}", binding_T=int(worst["T"]),
                            n=int(worst["n"]), rate=float(worst.solve_rate),
                            wilson_lo_simult=min(los),
                            clears_points=clears == nb,
                            clears_at_95=bool(clears == nb and min(los) >= GATE)))
    return pd.DataFrame(out)

aud = pd.concat([gate_audit(dep[dep.seed == 0], "n_pool 1500"),
                 gate_audit(big, "n_pool 6000")], ignore_index=True)
print(aud.round(4).to_string(index=False))

for rho in sorted(aud.rho.unique(), reverse=True):
    for src in ("n_pool 1500", "n_pool 6000"):
        c = aud[(aud.rho == rho) & (aud.source == src)]
        pts = c[c.clears_points]; safe = c[c.clears_at_95]
        f = lambda d: (min(int(x.split("=")[1].split(" ")[0]) for x in d.cell) if len(d) else None)
        print(f"rho={rho:.2f}  {src}:  L* (points) = {f(pts)},  L* (simultaneous 95%) = {f(safe)}")

## 8. Comparison to the previous best, same eval

Previous best: **L=12, seed 0, job `50527633`, finding 38** -- `0.9780` on the shared n=1500 pool.
Everything below is scored on the identical 1500 instances, paired.

In [ ]:
TONIGHT = [k for k in ("L=8 s1", "L=10 s1") if k in solved]
prev = "L=12 s0"
print(f"previous best  {prev:<8} {solved[prev].mean():.4f}  "
      f"[{wilson(int(solved[prev].sum()), 1500)[0]:.4f}, "
      f"{wilson(int(solved[prev].sum()), 1500)[1]:.4f}]  (finding 38, job 50527633)")
# "best tonight" is the best of the cells TRAINED tonight, not of the file --
# the file's best is `prev`, and conflating them would read as an improvement.
bt = max(TONIGHT, key=lambda k: solved[k].mean())
print(f"best tonight   {bt:<8} {solved[bt].mean():.4f}")
rows = []
for k in TONIGHT:
    d, w, l, p = paired(k, prev)
    twin = k.split()[0] + " s0"
    dt = paired(k, twin)[0] if twin in solved else float("nan")
    rows.append(dict(new_cell=k, rate=solved[k].mean(), vs_prev_best=d,
                     discordant=f"{w}-{l}", mcnemar_p=f"{p:.2e}",
                     vs_own_seed0_twin=dt,
                     verdict="improvement" if d > 0 and p < 0.05 else "no improvement"))
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nNo cell trained tonight improves on the previous best. Note the two sit on OPPOSITE sides")
print("of their own seed-0 twins, so the seed is not a bias one could subtract: it is noise with")
print("a sign that changes from rung to rung. The result is the error bar, not a new number.")

## 9. Interpretation, and what it costs the plan

**H1 is confirmed and over-confirmed.** Finding 40's inversion was optimization noise -- and the
cleanest demonstration is that re-running the same comparison inside seed 1 *reverses its sign*
(+0.1920 against -0.0653). **Finding 40 is withdrawn.** But the noise is so large that it also
swallows the signal the ladder was built to measure. The mechanism is the one pre-registered in
(c): near this knee, solve rate is extraordinarily steep in `val_nll` -- about **25 solve-rate
points per 0.01 of nll** -- so a seed that moves final `val_nll` by an unremarkable 0.005 moves
solve rate by a great deal.

**What this does NOT threaten.** The seed moves a rung, not the project:

- L=12 (0.9780) beats L=3 (0.0093) by 0.97 and L=6 (0.6153) by 0.36 -- gaps several times the seed
  spread. **Depth matters at N=48.** What is unresolved is the exact rung at which it stops
  mattering, which is precisely what L\* names.
- Findings 27/28/29 -- the DD verdict from SAE features, ablations and interventions -- are
  interventions on *fixed checkpoints* and are untouched by any of this.
- The adjacency-order leak fix (finding 16) and the depth-bin construction are unaffected.

**What it costs.** The MLP-depth scan hshunt asked for has L\* as its dependent variable. L\* now
carries two error bars: sampling, measured and controlled tonight, and seed, measured and *not*
controlled. A scan over MLP depth whose rungs each carry a single draw cannot resolve a one-rung
effect against the second.

**The cheapest fix, and it is not more rungs.** Three seeds at **L=8 and L=12 only** -- the two
cells that bracket L\* at rho=0.60 -- give a standard error of roughly 0.09 per rung and answer
whether L\*(48) is stable under reseeding at all. That is 4 more jobs at 7-11 h each, against 6 new
single-draw rungs. Measure the error bar before adding points to the curve.

## 10. Limitations

1. **Two seeds, not three.** Two draws bound a spread; they do not estimate a variance. Every
   "spread" number here is a range of two, and the SE-per-rung figure in §9 is a planning estimate.
   The two spreads here even have opposite signs (seed 1 worse at L=8, better at L=10), so nothing
   here licenses treating either seed as systematically stronger.
2. **Only L=8 and L=10 replicated.** L=12, the current best and the top of the bracket, still has
   one seed. If its draw was lucky the whole bracket shifts.
3. **239,999 of 240,000 training instances shared** between the seeds (finding 23). This isolates
   init and batch order, which is what the ladder's rungs vary -- but it is not a full data-resample
   error bar, and the true run-to-run spread for an independently sampled training set could be
   larger.
4. **`50728837` exited 1:0.** The failure was in a post-hoc pandas block, after all 60 rows were
   written; the eval itself completed. Documented rather than re-run.
5. **Fast phase only** (finding 34) and **ff=1024 throughout** (finding 14): every number here is
   "at ff=1024, in the log N regime".
6. **L\* at a simultaneous 95%** is a conservative reading. Bonferroni over bins is not tight, and a
   less conservative multiplicity correction would move some verdicts; the point stands that the
   point estimates were being quoted with precision they do not have.